# E09 · Pose graphs, loop closure and drift

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/06_estimation_perception/E09_pose_graphs_loop_closure_and_drift.ipynb)

**Track:** Estimation and perception · **Skill axis (sheet):** Estimation and perception · **Time:** about 60 min

**Skill Hub concepts:** Simultaneous localization and mapping; Loop closure

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("E09")

## Map: SLAM as least squares on a graph

**Nodes** are robot poses $p_0 \dots p_{n-1}$; **edges** are relative measurements $z_{ij} \approx p_j - p_i$ with a
weight $w_{ij}$ (inverse variance). Odometry gives edges between consecutive poses; a **loop closure** (place
recognition, scan matching) adds an edge between poses far apart in time. Graph SLAM back end:
$$\hat p = \arg\min_p \sum_{(i,j)} w_{ij} \left\lVert p_j - p_i - z_{ij} \right\rVert^2 \quad \text{with } p_0 \text{ fixed (the gauge)}$$
With positions only (this lab) the problem is **linear**: stack rows $\sqrt{w_{ij}}\,(p_j - p_i) = \sqrt{w_{ij}}\,z_{ij}$
and solve with least squares. With headings ($SE(2)$) it becomes nonlinear and you iterate Gauss-Newton (E02) on the
same sparse structure (g2o, GTSAM, Ceres, slam_toolbox).

```
 p3 ── p4 ── p5            odometry edges: drift accumulates along the chain
 │            │            loop closure p9→p0: one edge that pulls the whole chain back
 p2          p6            false loop closure: bends the whole map. Gate it with the
 │            │            Mahalanobis distance before accepting, and use robust kernels.
 p1   p0 ── p9 ── p8 ── p7
```

| Front end (sensors) | Back end (optimization) |
|---|---|
| odometry, scan matching, feature tracking | build the graph, solve least squares |
| place recognition proposes loop closures | gating and robust kernels reject wrong ones |

In [ ]:
import numpy as np

## Task 1: build and solve the linear pose graph
`edges` is a list of `(i, j, z, w)` with `z` a length-2 array. Unknown vector: $[x_0, y_0, x_1, y_1, \dots]$.
Fix $p_0 = (0, 0)$ with two extra rows of weight `w_anchor`. Return `A` (rows = 2 per edge + 2) and `b`.

In [ ]:
def build_system(n, edges, w_anchor=1e6):
    raise NotImplementedError  # TODO

def solve_graph(n, edges):
    """Return positions as an (n, 2) array."""
    raise NotImplementedError  # TODO

lab.check("graph", build_system, solve_graph)

## Task 2: gate loop closure candidates
Dead-reckoned estimate `est` (n x 2). Odometry noise per edge has standard deviation `sigma` in each axis, so the
predicted relative position $\hat p_j - \hat p_i$ has covariance $S = (|j - i| + 1)\,\sigma^2 I$ (chain plus the loop
measurement). Return `(d2, accept)` with $d^2 = r^\top S^{-1} r$, $r = z - (\hat p_j - \hat p_i)$, accept when
$d^2 < 5.991$ (95 % chi-square, 2 degrees of freedom).

In [ ]:
def gate(est, i, j, z, sigma):
    raise NotImplementedError  # TODO

lab.check("gate", gate)

In [ ]:
try:
    import matplotlib.pyplot as plt
    truth = np.array([[0, 0], [1, 0], [2, 0], [3, 0], [3, 1], [3, 2], [2, 2], [1, 2], [0, 2], [0, 1]], float)
    rng = np.random.default_rng(3)
    odo = [(k, k + 1, truth[k + 1] - truth[k] + np.array([0.04, 0.03]) + rng.normal(0, 0.02, 2), 400.0) for k in range(9)]
    dead = solve_graph(10, odo)
    fixed = solve_graph(10, odo + [(9, 0, truth[0] - truth[9], 400.0)])
    for P, lbl in [(truth, "truth"), (dead, "odometry only"), (fixed, "with loop closure")]:
        plt.plot(*np.vstack([P, P[:1]]).T, "o-", label=lbl)
    plt.axis("equal"); plt.legend(); plt.show()
except NotImplementedError:
    print("finish Task 1 first")

## Checkpoint: Math
**M1 (Apply).** Unknowns in a positions-only 2D pose graph with 10 poses (before anchoring)?

**M2 (Apply).** Edges for 10 poses chained by odometry plus one loop closure?

**M3 (Remember).** Chi-square threshold used to gate a 2D residual at 95 %?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** You add a wrong loop closure with the same weight as the odometry edges. What happens?
- a) The optimizer ignores it because the other edges outvote it
- b) The error spreads over the whole loop: every pose shifts and the map bends
- c) Only the two poses on that edge move

**K2 (Understand).** Explain why odometry-only uncertainty grows along the trajectory and how one correct loop closure
reduces the error everywhere in the loop, not only at its two ends.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `e09.h`).
3. In `roboprep/checks/e09.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()